# MMseqs2 Homology Search

Generate Multiple Sequence Alignments (MSAs) for protein sequences against MMseqs2-indexed databases. Two search modes:

- **`search_mode="remote"` (default)** — queries the ColabFold MSA API; no local DB / GPU. Convenient for one-offs and onboarding.
- **`search_mode="local"`** — runs MMseqs2 against a registry-provisioned DB on disk; GPU-accelerated by default via [MMseqs2-GPU](https://www.nature.com/articles/s41592-025-02819-8). Higher throughput, no rate limits.

Singleton queries return unpaired MSAs; multi-chain groups return taxonomy-paired MSAs (row-aligned across chains) — direct input for AlphaFold3 / Boltz2 / Chai1 / Protenix / AlphaFold2.


In [1]:
from pathlib import Path

from proto_tools.tools.sequence_alignment.mmseqs2 import (
    Mmseqs2HomologySearchConfig,
    Mmseqs2HomologySearchInput,
    run_mmseqs2_homology_search,
)


## Single-sequence remote search

Plain-string sugar: `queries=[seq]` becomes a singleton group with an auto-generated `sequence_id`. The default config hits the ColabFold MSA API — no local DB required.


In [2]:
ubiquitin = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

inputs = Mmseqs2HomologySearchInput(queries=[ubiquitin])
config = Mmseqs2HomologySearchConfig()  # search_mode="remote" by default
result = run_mmseqs2_homology_search(inputs, config)

group = result.results[0]
print(f"sequence_id: {group.sequence_ids[0]}")
print(f"datasets:    {group.datasets_searched}")
print(f"homologs:    {group.num_homologs_found[0]}")


Running run_mmseqs2_homology_search [00:00]

sequence_id: seq_233b4b0b8c
datasets:    ['colabfold-remote']
homologs:    9654


## Batch with explicit IDs

Pass `(sequence, id)` tuples (or `Mmseqs2HomologySearchQuery` objects) to attach stable identifiers — the same IDs show up in `result.sequence_ids` and in any A3M/FASTA exports.


In [3]:
batch = Mmseqs2HomologySearchInput(
    queries=[
        ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG", "hba_human"),
        ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK", "hbb_human"),
    ],
)
batch_result = run_mmseqs2_homology_search(batch, Mmseqs2HomologySearchConfig())
for grp in batch_result:
    print(f"{grp.sequence_ids[0]}: {grp.num_homologs_found[0]} homologs")


Running run_mmseqs2_homology_search [00:00]

hba_human: 3677 homologs
hbb_human: 3956 homologs


## Paired search (heterocomplex chains)

Nest a list of queries to mark them as one paired group. The chains are submitted together with `use_pairing=True`, and the per-chain MSAs come back **row-aligned by taxonomy** — row N of every chain is the same organism. Structure predictors use this cross-chain co-evolutionary signal to refine interface predictions.

Each result's `paired_msas` carries the row-aligned MSAs; `msas` carries the unpaired per-chain MSAs (always populated).


In [4]:
paired_inputs = Mmseqs2HomologySearchInput(
    queries=[
        [
            ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG", "hba"),
            ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK", "hbb"),
        ]
    ]
)
paired_result = run_mmseqs2_homology_search(paired_inputs, Mmseqs2HomologySearchConfig())

grp = paired_result.results[0]
for chain_id, paired_msa in zip(grp.sequence_ids, grp.paired_msas):
    n = paired_msa.num_sequences if paired_msa is not None else 0
    print(f"chain {chain_id}: {n} paired rows")


Running run_mmseqs2_homology_search [00:00]

chain hba: 1478 paired rows
chain hbb: 1478 paired rows


## Local search

Switch to `search_mode="local"` to search a registry-provisioned database on disk. Higher throughput, no rate limits, and supports the metagenomic envdb for deeper unpaired MSAs.

Provision a database with `python -m proto_tools.tools.sequence_alignment.mmseqs2.setup_databases uniref30-2302` (one-time, see the [tool README](../README.md) for sizing). Use `use_metagenomic_db=True` to also search `colabfold-envdb-202108` — it must be provisioned separately.


In [5]:
# Construct only — actually running requires a provisioned DB.
local_config = Mmseqs2HomologySearchConfig(
    search_mode="local",
    dataset="uniref30-2302",
    use_gpu=False,           # set True to use the GPU-padded index
    use_metagenomic_db=False,  # set True to also search the ColabFold envdb
)
print(f"mode:           {local_config.search_mode}")
print(f"dataset:        {local_config.dataset}")
print(f"GPUs reserved:  {local_config.gpus_per_instance}")


mode:           local
dataset:        uniref30-2302
GPUs reserved:  0


## Inspect the MSA

Each `MSA` exposes alignment stats and indexable rows. The number of sequences includes the query itself, so homolog count is one less than the total. Alignment length matches the query length (all hits are aligned to the query frame). The average gap fraction indicates how divergent the homologs are.


In [6]:
msa = batch_result.results[0].msas[0]
print(f"sequences:           {msa.num_sequences}")
print(f"alignment length:    {msa.alignment_length}")
print(f"avg gap fraction:    {msa.average_gap_fraction:.3f}")
print()
print("Query (first entry):")
print(f"  ID:  {msa.sequence_ids[0]}")
print(f"  Seq: {msa[0][:60]}...")


sequences:           3678
alignment length:    60
avg gap fraction:    0.063

Query (first entry):
  ID:  hba_human
  Seq: MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG...


## Export results

MSA results export to standard alignment formats. **A3M** is the compressed format that ColabFold / AlphaFold / Boltz / Chai accept directly. **FASTA** is widely compatible for general sequence-analysis tools. Files are named by the query's `sequence_id`; paired groups get one file per chain.


In [7]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export("mmseqs2_a3m", export_path=output_dir, file_format="a3m")
print("A3M files exported to ./example_output/mmseqs2_a3m/")

batch_result.export("mmseqs2_fasta", export_path=output_dir, file_format="fasta")
print("FASTA files exported to ./example_output/mmseqs2_fasta/")


A3M files exported to ./example_output/mmseqs2_a3m/
FASTA files exported to ./example_output/mmseqs2_fasta/
